## NOTEBOOK GOLD + ML
Business analytics + predicción + consumo SQL

In [0]:
# Cargamos tablas silver
prices = spark.table("workspace.portfolio_intel.silver_prices_clean")
returns = spark.table("workspace.portfolio_intel.silver_returns")
assets = spark.table("workspace.portfolio_intel.bronze_assets")


## 1 Construir valor del portfolio diario
* Precio × peso
* Suma por día
* Métrica clara de negocio

In [0]:
from pyspark.sql.functions import (
    col, lag, log, exp, when, current_timestamp,
    avg, stddev, max as spark_max, min as spark_min, sum as spark_sum, lit
)
from pyspark.sql.window import Window

In [0]:

portfolio_daily = (
    prices
    .join(assets, on="symbol", how="inner")
    .select(
        "date", "symbol", "price", "weight_target"
    )
    .withColumn(
        "weighted_price",
        col("price") * col("weight_target")
    )
    .groupBy("date")
    .agg(
        spark_sum("weighted_price").alias("portfolio_value")
    )
)
display(portfolio_daily)

date,portfolio_value
2024-10-26,6701.469531250001
2024-11-04,7239.773881530762
2024-11-20,9907.389239501954
2024-06-25,6618.060284423828
2024-12-20,10253.418533325197
2024-11-24,9801.382031250001
2024-09-05,6054.602072143554
2025-09-07,11116.76171875
2026-01-04,9141.34921875
2025-05-03,9589.1796875


In [0]:
#Retornos del portfolio
w = Window.orderBy("date")

portfolio_daily = (
    portfolio_daily
    .withColumn("prev_value", lag("portfolio_value").over(w))
    .withColumn(
        "daily_return",
        log(col("portfolio_value") / col("prev_value"))
    )
    .withColumn("cumulative_return", spark_sum("daily_return").over(w))
    .withColumn("created_ts", current_timestamp())
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
portfolio_daily = portfolio_daily.drop("prev_value")
portfolio_daily.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.gold_portfolio_daily"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## 3 Métricas de riesgo ejecutivas




In [0]:
risk_metrics = (
    portfolio_daily
    .agg(
        (spark_sum("daily_return") / stddev("daily_return")).alias("sharpe_ratio"),
        spark_min("daily_return").alias("worst_day"),
        stddev("daily_return").alias("volatility")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
risk_metrics_df = spark.createDataFrame(
    [
        ("Sharpe Ratio", risk_metrics.collect()[0]["sharpe_ratio"]),
        ("Worst Daily Return", risk_metrics.collect()[0]["worst_day"]),
        ("Volatility", risk_metrics.collect()[0]["volatility"]),
    ],
    ["metric_name", "metric_value"]
).withColumn("computed_ts", current_timestamp())


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
risk_metrics_df.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.gold_risk_metrics"
)


## 4 Contribución por activo

In [0]:
asset_contrib = (
    prices
    .join(returns, on=["symbol", "date"])
    .join(assets, on="symbol")
    .select(
        "date", "symbol",
        "return_1d", "weight_target"
    )
    .withColumn("contribution", col("return_1d") * col("weight_target"))
    .withColumn("created_ts", current_timestamp())
)


In [0]:
asset_contrib_fixed = (
    asset_contrib
    .select(
        "symbol",
        "date",
        col("weight_target").alias("weight"),
        "contribution",
        "created_ts"
    )
)
asset_contrib_fixed.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.gold_asset_contribution"
)

## Machine learning. 

No estaremos prediciendo mercados aquí. Es solo una demostración de una arquitectura ML productiva.

Entrenaremos un Logistic Regression Model

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
import mlflow

features = spark.table("workspace.portfolio_intel.silver_features_ml")



In [0]:
assembler = VectorAssembler(
    inputCols=[
        "return_1d", "return_5d",
        "rolling_mean_20", "rolling_vol_20",
        "momentum_10"
    ],
    outputCol="features"
)


In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="target_up_1d",
    probabilityCol="probability"
)


In [0]:
pipeline = Pipeline(stages=[assembler, lr])


In [0]:
%sql
CREATE VOLUME workspace.portfolio_intel.mlflow_tmp;

In [0]:
%sql

SHOW VOLUMES IN workspace.portfolio_intel;

database,volume_name
portfolio_intel,mlflow_tmp


In [0]:
with mlflow.start_run():
    model = pipeline.fit(features)
    mlflow.spark.log_model(
        model,
        "portfolio_direction_model",
        dfs_tmpdir="/Volumes/workspace/portfolio_intel/mlflow_tmp/"
    )

2026/01/09 19:59:49 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/09 19:59:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-98e86dc2-1b99-4d06-8cf7-91/tmp9chha7po/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/01/09 19:59:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [0]:
### Predicciones y persistencia GOLD
predictions = model.transform(features)


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def extract_prob_up(probability):
    # probability.values is an array of probabilities for each class
    return float(probability.values[1])

extract_prob_up_udf = udf(extract_prob_up, DoubleType())

final_preds = (
    predictions
    .select(
        "symbol",
        "date",
        extract_prob_up_udf(col("probability")).alias("prob_up")
    )
    .withColumn("model_version", lit("v1"))
    .withColumn("prediction_ts", current_timestamp())
)


In [0]:
final_preds.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.gold_return_predictions"
)


###Spark ML → MLflow → Delta → SQL → Dashboard